In [1]:
import sys
sys.path.insert(0, '/home/joseluisalmendarezgonzalez/Desktop/3_GEO_FNO')

In [2]:
import torch
import random
import numpy as np
from torch.utils.data import DataLoader, random_split
from lib.Common import setup_logging, KolmogorovDataset, FNOGenerator
from lib.Lebesgue2Approach import FNOSupervisedTrainer

In [3]:
from torch.cuda.amp import autocast, GradScaler

In [4]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [5]:
setup_logging("l2_experiment.log")

In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_PATH = "../Dataset/snapshots_64x64_use.npy"

In [7]:
dataset = KolmogorovDataset(DATA_PATH, seq_len=10)
train_len = int(0.8 * len(dataset))
val_len = len(dataset) - train_len
train_ds, val_ds = random_split(dataset, [train_len, val_len])

2026-06-07 23:44:21,546 | INFO | Dataset: 1152 trayectorias × 90 ventanas = 103,680 muestras | seq_len=10 | H×W=64×64


In [8]:
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=4)

In [9]:
G = FNOGenerator(hidden_ch=32, modes1=12, modes2=12, n_layers=4, z_dim=0).to(DEVICE)

In [12]:
trainer = FNOSupervisedTrainer(
    G, DEVICE,
    lr=1e-4,
    log_dir="logs_mse",
    resume=True,
    patience=5
)

2026-06-07 23:45:03,390 | WARNING | No se encontró checkpoint; empezando desde cero.


In [13]:
history = trainer.fit(train_loader, val_loader, epochs=50)
print(history)

Training MSE:   2%|▋                             | 239/10368 [00:54<38:21,  4.40it/s, loss=0.03231]


KeyboardInterrupt: 